# P6 – Table de la période longue (prix 2017 à 2025)
**Contributeur :** Théophyle · **Date :** 06/11/2026

**Pourquoi :** le tour 2 apprend sur plusieurs phases du marché (stabilité 2017-2018, hausse 2019-2022, recul 2023) au lieu de la seule hausse 2020-2022. H4, H5, H6 et H7 en dépendent.

**Constat :** `table_modelisation_bv.parquet` couvre déjà les prix 2015 à 2025, avec le même seuil de 20 ventes et les mêmes colonnes. La délinquance ne commence qu'avec les prix 2017 : le masque commun de `charger_table()` retire 2015 et 2016 et garde 2017-2025. La période est un paramètre de `commun.py` : `fenetres(table, annee_debut=2017)` (P5). Il n'y a donc pas de nouvelle table : ce notebook vérifie les conditions de la feuille de route.

In [1]:
import pandas as pd
import commun
from commun import charger_table, decouper, fenetres

ANNEE_DEBUT_LONGUE = 2017            # premiers prix avec la délinquance (feuille de route, P6)

brute = charger_table(masquer=False)  # toutes les lignes, avant le masque commun
table = charger_table()               # lignes sans case vide : celles des modèles
print("Table brute :", brute.shape, "| après le masque :", table.shape)

Table brute : (28450, 39) | après le masque : (23489, 39)


## 1. Lignes par année et type de bien
Avant et après le masque commun (lignes à case vide retirées une fois pour toutes).

In [2]:
lignes = pd.DataFrame({"lignes (table brute)": brute.groupby(level=["annee", "type_bien"]).size(),
                       "lignes gardées (masque commun)": table.groupby(level=["annee", "type_bien"]).size()})
lignes.unstack()

lignes (table brute)        lignes gardées (masque commun)        
type_bien          appartement maison                    appartement  maison
annee                                                                       
2015                       691   1769                            NaN     NaN
2016                       717   1775                            NaN     NaN
2017                       746   1776                          746.0  1775.0
2018                       774   1780                          774.0  1779.0
2019                       805   1780                          805.0  1779.0
2020                       857   1780                          857.0  1779.0
2021                       878   1781                          878.0  1780.0
2022                       912   1782                          912.0  1781.0
2023                       861   1781                          861.0  1780.0
2024                       823   1777                          823.0  1776.0
2025                       827   1778                          827.0  1777.0

## 2. Part de cases vides par variable et par année
Table brute. Seules les variables qui ont au moins une case vide sont affichées.

In [3]:
vides = brute.drop(columns=["annee"]).isna().groupby(level="annee").mean().mul(100).round(1)
vides.loc[:, vides.max() > 0]

,part_indicateurs_estimes,taux3_vols_vehicules,taux3_vols_violents,taux3_vols_sans_violence,taux3_cambriolages,taux3_stupefiants,taux3_violences_intrafamiliales,taux3_violences_sexuelles
annee,,,,,,,,
2015,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2016,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Fenêtres de la période longue
Mêmes années jugées (2023 et 2024), apprentissage à partir de 2017.

In [4]:
for annee, (app, ev) in fenetres(table, annee_debut=ANNEE_DEBUT_LONGUE).items():
    annees_app = sorted(int(a) for a in app["annee"].unique())
    print("Fenêtre", annee, ": apprentissage", annees_app, "(", len(app), "lignes ) | jugée", annee, "(", len(ev), "lignes )")

Fenêtre 2023 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022] ( 15645 lignes ) | jugée 2023 ( 2641 lignes )
Fenêtre 2024 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022, 2023] ( 18286 lignes ) | jugée 2024 ( 2599 lignes )


## 4. Identique à la table actuelle sur les années communes
L'apprentissage long, limité aux années 2020 et suivantes, doit être exactement l'apprentissage actuel.

In [5]:
for annee in commun.ANNEES_JUGEES:
    app_actuel, _ = decouper(table, annee_eval=annee)
    app_long, _ = decouper(table, annee_eval=annee, annee_debut=ANNEE_DEBUT_LONGUE)
    identique = app_long[app_long["annee"] >= commun.ANNEE_DEBUT].equals(app_actuel)
    print("Fenêtre", annee, ": identique sur les années communes ->", identique)
    assert identique

Fenêtre 2023 : identique sur les années communes -> True
Fenêtre 2024 : identique sur les années communes -> True


## 5. Variables fixes et variables qui changent chaque année
Pour chaque variable du territoire, le nombre maximal de valeurs différentes d'un même sous-bassin sur 2017-2025. 1 = une photo fixe, appliquée à toutes les années (recensement 2022, équipements 2025, Filosofi, LOVAC) ; plus de 1 = une variable qui change chaque année.

In [6]:
territoire = table.xs("maison", level="type_bien")      # les variables du territoire sont les mêmes pour les deux types
HORS = ["annee", "maison", "prix", "prix_n1", "n_ventes", "nb_ventes_n1", "BV2022"]
valeurs_differentes = territoire.drop(columns=HORS).groupby(level="sous_bassin_de_vie").nunique().max()
valeurs_differentes.sort_values().to_frame("valeurs différentes au plus (2017-2025)")

,valeurs différentes au plus (2017-2025)
taux_equip_transports,1
part_apparts,1
part_rsec,1
taux_vacance,1
densite_population,1
part_avant1946,1
taux_equip_tourisme,1
taux_equip_sports,1
taux_equip_services,1
taux_equip_sante,1


## Conclusion
- **La période longue (prix 2017 à 2025) est disponible dans la table actuelle** : `fenetres(charger_table(), annee_debut=2017)`. Pas de nouvelle table : même seuil de 20 ventes, mêmes colonnes.
- **Identique à la table actuelle sur les années communes** : l'apprentissage long, limité aux années 2020 et suivantes, est exactement l'apprentissage actuel, pour les deux fenêtres (section 4).
- **Lignes** : le masque commun retire 2015 et 2016 (délinquance absente, 100 % de cases vides) ; à partir de 2017, de 1 775 à 1 781 lignes de maisons et de 746 à 912 lignes d'appartements par an (section 1). Une ligne de maison par an est aussi retirée : le sous-bassin 49069, sans donnée LOVAC.
- **Fenêtres** : apprentissage 2017-2022 (15 645 lignes) pour juger 2023, 2017-2023 (18 286 lignes) pour juger 2024 (section 3), contre 7 987 et 10 628 lignes avec l'apprentissage actuel.
- **Variables qui changent chaque année** (section 5) : la délinquance (une valeur par année) et les ménages (recensements 2019 à 2022, puis estimation 2024 ; le recensement 2019 sert pour les prix 2017 à 2019). Toutes les autres variables du territoire sont des photos fixes.
- **Limite à déclarer** : les photos fixes (recensement 2022, équipements 2025, Filosofi, LOVAC) s'appliquent aussi aux prix 2017-2019 : pour ces années, on utilise une information plus récente. C'est accepté par la feuille de route (« photos fixes étendues aux années anciennes ») ; le niveau 2 mise de toute façon sur les variables qui changent chaque année.